In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Path to cleaned datasets
DATA_PATH = "../outputs/cleaned_data/"

# Load cleaned datasets
competitions = pd.read_csv(DATA_PATH + "competitions_cleaned.csv")
teams = pd.read_csv(DATA_PATH + "teams_cleaned.csv")
matches = pd.read_csv(DATA_PATH + "matches_cleaned.csv")
ball_by_ball = pd.read_csv(DATA_PATH + "ball_by_ball_cleaned.csv")
player_mapping = pd.read_csv(DATA_PATH + "player_mapping_cleaned.csv")
playing_eleven = pd.read_csv(DATA_PATH + "playing_eleven_cleaned.csv")

# Convert match date
matches["match_date"] = pd.to_datetime(matches["match_date"])

# Display dataset information
datasets = {
    "competitions": competitions,
    "teams": teams,
    "matches": matches,
    "ball_by_ball": ball_by_ball,
    "player_mapping": player_mapping,
    "playing_eleven": playing_eleven
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

competitions: (141, 4)
teams: (353, 3)
matches: (883, 22)
ball_by_ball: (200999, 33)
player_mapping: (6413, 6)
playing_eleven: (28381, 9)


In [2]:
# COMPETITION AND SEASON ANALYSIS

print("TOTAL COMPETITIONS:", competitions["comp_id"].nunique())

print("\nMATCH FORMATS")
display(competitions["match_format"].value_counts())

print("\nSEASON DISTRIBUTION")
display(
    competitions.groupby("season")
    .agg(
        total_competitions=("comp_id", "nunique"),
        match_formats=("match_format", "nunique")
    )
    .sort_index()
)

print("\nCOMPETITION DETAILS")
display(competitions.head(10))

TOTAL COMPETITIONS: 141

MATCH FORMATS


match_format
T20I    134
T20       7
Name: count, dtype: int64


SEASON DISTRIBUTION


,total_competitions,match_formats
season,,
2022,25,2
2022/23,3,1
2023,20,2
2023/24,1,1
2024,33,2
2024/25,1,1
2025,33,2
2026,25,2



COMPETITION DETAILS


,comp_id,comp_name,match_format,season
0,3395,South Africa in New Zealand T20I Series 2026,T20I,2026
1,3405,Afghanistan v Sri Lanka T20I Series 2026,T20I,2026
2,3465,India in Zimbabwe T20I Series 2026,T20I,2026
3,3568,Bangladesh in Zimbabwe T20I Series 2026,T20I,2026
4,3482,Australia in Bangladesh T20I Series 2026,T20I,2026
5,3464,Sri Lanka in West Indies T20I Series 2026,T20I,2026
6,3765,Sri Lanka in Pakistan T20I Series 2026,T20I,2026
7,3739,India in Japan T20I Match 2026,T20I,2026
8,3787,Afghanistan v Zimbabwe T20I Series 2026,T20I,2026
9,3713,Afghanistan vs India in India T20I Series 2026,T20I,2026


In [3]:
# MATCH DISTRIBUTION BY SEASON

match_analysis = matches.copy()

# Connect matches with competition information
match_analysis = match_analysis.merge(
    competitions[["comp_id", "comp_name", "match_format", "season"]],
    on="comp_id",
    how="left",
    validate="many_to_one"
)

print("TOTAL MATCHES:", match_analysis["match_id"].nunique())

print("\nMATCHES BY SEASON")
display(
    match_analysis.groupby("season")
    .agg(
        total_matches=("match_id", "nunique"),
        competitions=("comp_id", "nunique")
    )
    .sort_index()
)

print("\nMATCHES BY FORMAT")
display(
    match_analysis.groupby("match_format")
    .agg(
        total_matches=("match_id", "nunique"),
        seasons=("season", "nunique")
    )
)

TOTAL MATCHES: 883

MATCHES BY SEASON


,total_matches,competitions
season,,
2022,168,25
2022/23,12,3
2023,137,20
2023/24,3,1
2024,198,33
2024/25,3,1
2025,216,32
2026,146,23



MATCHES BY FORMAT


,total_matches,seasons
match_format,,
T20,401,5
T20I,482,8


In [4]:
# MATCH DISTRIBUTION BY COMPETITION

competition_match_summary = (
    match_analysis.groupby(
        ["comp_name", "match_format", "season"],
        dropna=False
    )
    .agg(
        total_matches=("match_id", "nunique")
    )
    .reset_index()
    .sort_values(
        ["match_format", "season", "total_matches"],
        ascending=[True, True, False]
    )
)

display(competition_match_summary.to_string(index=False))

'                                              comp_name match_format  season  total_matches\n                                               IPL 2022          T20    2022             74\n                                               IPL 2023          T20    2023             74\n                                               IPL 2024          T20    2024             74\n                  Mens T20 Emerging Teams Asia Cup 2024          T20    2024             15\n                                               IPL 2025          T20    2025             75\n                             Asia Cup Rising Stars 2025          T20    2025             15\n                                               IPL 2026          T20    2026             74\n                            Mens Twenty20 Asia Cup 2022         T20I    2022             13\n                   England in Pakistan T20I Series 2022         T20I    2022              7\n                           Asia Cup T20 Qualifiers 2022         T20I 

In [5]:
# IDENTIFY IPL MATCHES

ipl_matches = match_analysis[
    match_analysis["comp_name"].str.fullmatch(
        r"IPL \d{4}",
        na=False
    )
].copy()

print("IPL MATCHES:", ipl_matches["match_id"].nunique())

print("\nIPL MATCHES BY SEASON:")
display(
    ipl_matches.groupby("season")
    .agg(
        total_matches=("match_id", "nunique")
    )
    .sort_index()
)

print("\nOTHER T20 COMPETITIONS:")
display(
    match_analysis[
        (match_analysis["match_format"] == "T20") &
        (~match_analysis["comp_name"].str.fullmatch(r"IPL \d{4}", na=False))
    ][["comp_name", "season"]]
    .drop_duplicates()
    .sort_values(["season", "comp_name"])
)

IPL MATCHES: 371

IPL MATCHES BY SEASON:


,total_matches
season,
2022,74
2023,74
2024,74
2025,75
2026,74



OTHER T20 COMPETITIONS:


,comp_name,season
499,Mens T20 Emerging Teams Asia Cup 2024,2024
733,Asia Cup Rising Stars 2025,2025


In [6]:
# IPL MATCH RESULT SUMMARY

print("IPL MATCH RESULT STATUS:")
display(ipl_matches["result"].value_counts(dropna=False))

print("\nIPL WINNER INFORMATION:")
print("Missing winner IDs:", ipl_matches["win_team_id"].isna().sum())
print("Matches with winner IDs:", ipl_matches["win_team_id"].notna().sum())

print("\nIPL MATCHES BY RESULT TYPE:")
display(
    ipl_matches.groupby(["season", "result"], dropna=False)
    .size()
    .unstack(fill_value=0)
)

IPL MATCH RESULT STATUS:


result
Delhi Capitals Won by 6 wickets               7
Kolkata Knight Riders Won by 8 wickets        6
Punjab Kings Won by 5 wickets                 5
Match Abandoned                               5
Gujarat Titans Won by 7 wickets               5
                                             ..
Sunrisers Hyderabad Won by 55 runs            1
Rajasthan Royals Won by 30 runs               1
Delhi Capitals Won by 40 runs                 1
Royal Challengers Bengaluru Won by 92 runs    1
Rajasthan Royals Won by 47 runs               1
Name: count, Length: 283, dtype: int64


IPL WINNER INFORMATION:
Missing winner IDs: 9
Matches with winner IDs: 362

IPL MATCHES BY RESULT TYPE:


result,Chennai Super Kings Won by 103 runs,Chennai Super Kings Won by 12 runs,Chennai Super Kings Won by 15 runs,Chennai Super Kings Won by 2 wickets,Chennai Super Kings Won by 20 runs,Chennai Super Kings Won by 23 Runs,Chennai Super Kings Won by 23 runs,Chennai Super Kings Won by 27 Runs,Chennai Super Kings Won by 28 runs,Chennai Super Kings Won by 3 Wickets,...,Sunrisers Hyderabad Won by 65 runs,Sunrisers Hyderabad Won by 67 runs,Sunrisers Hyderabad Won by 8 Wickets,Sunrisers Hyderabad Won by 8 wickets,Sunrisers Hyderabad Won by 9 Runs,Sunrisers Hyderabad won by 31 runs,Sunrisers Hyderabad won by 7 wicket,Sunrisers Hyderabad won by 7 wickets,Sunrisers Hyderabad won by 8 wickets,Sunrisers Hyderabad won by 9 wickets
season,,,,,,,,,,,,,,,,,,,,,
2022,0,0,0,0,0,1,0,0,0,1,...,0,0,1,0,0,0,1,1,1,1
2023,0,1,1,0,0,0,0,1,0,0,...,0,0,0,0,1,0,0,0,1,0
2024,0,0,0,0,1,0,0,0,1,0,...,0,1,0,0,0,1,0,0,0,0
2025,0,0,0,1,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
2026,1,0,0,0,0,0,1,0,0,0,...,1,0,0,0,0,0,0,0,0,0


In [7]:
# IPL MATCHES WITHOUT A RECORDED WINNER

no_winner_matches = ipl_matches[
    ipl_matches["win_team_id"].isna()
].copy()

columns_to_check = [
    "match_id",
    "season",
    "match_date",
    "comp_name",
    "team1_id",
    "team2_id",
    "result",
    "win_team_id",
    "win_margin",
    "win_outcome"
]

display(
    no_winner_matches[
        [col for col in columns_to_check if col in no_winner_matches.columns]
    ].sort_values("match_date")
)

,match_id,season,match_date,comp_name,result,win_team_id
234,20226,2023,2023-05-03,IPL 2023,Match Abandoned,NaN
396,31281,2024,2024-05-13,IPL 2024,Match Abandoned,NaN
399,31284,2024,2024-05-16,IPL 2024,Match Abandoned,NaN
403,31288,2024,2024-05-19,IPL 2024,Match Abandoned,NaN
569,40737,2025,2025-04-26,IPL 2025,No Result,NaN
580,40748,2025,2025-05-05,IPL 2025,No Result,NaN
583,40751,2025,2025-05-08,IPL 2025,No Result,NaN
593,40761,2025,2025-05-17,IPL 2025,Match Abandoned,NaN
759,50136,2026,2026-04-06,IPL 2026,No Result,NaN


In [8]:
# IPL DATE COVERAGE

print("FIRST IPL MATCH:", ipl_matches["match_date"].min())
print("LAST IPL MATCH:", ipl_matches["match_date"].max())

print("\nMATCHES BY YEAR:")
display(
    ipl_matches.groupby(ipl_matches["match_date"].dt.year)
    .agg(
        total_matches=("match_id", "nunique"),
        first_match=("match_date", "min"),
        last_match=("match_date", "max")
    )
)

FIRST IPL MATCH: 2022-03-26 00:00:00
LAST IPL MATCH: 2026-05-31 00:00:00

MATCHES BY YEAR:


,total_matches,first_match,last_match
match_date,,,
2022,74,2022-03-26,2022-05-29
2023,74,2023-03-31,2023-05-29
2024,74,2024-03-22,2024-05-26
2025,75,2025-03-22,2025-06-03
2026,74,2026-03-28,2026-05-31


In [9]:
# IPL INNINGS DISTRIBUTION

ipl_deliveries = ball_by_ball[
    ball_by_ball["match_id"].isin(ipl_matches["match_id"])
].copy()

print("TOTAL IPL DELIVERIES:", len(ipl_deliveries))
print("\nINNINGS DISTRIBUTION:")

display(
    ipl_deliveries.groupby("innings")
    .agg(
        total_deliveries=("match_id", "size"),
        matches=("match_id", "nunique")
    )
    .sort_index()
)

print("\nMATCHES WITH MORE THAN TWO INNINGS:")

innings_per_match = (
    ipl_deliveries.groupby("match_id")["innings"]
    .nunique()
)

display(innings_per_match[innings_per_match > 2])

TOTAL IPL DELIVERIES: 87687

INNINGS DISTRIBUTION:


,total_deliveries,matches
innings,,
1,45588,367
2,42085,363
3,9,2
4,5,2



MATCHES WITH MORE THAN TWO INNINGS:


match_id
40725    4
50359    4
Name: innings, dtype: int64

In [10]:
# INSPECT IPL MATCHES WITH SUPER-OVER INNINGS

super_over_match_ids = [40725, 50359]

display(
    ipl_deliveries[
        ipl_deliveries["match_id"].isin(super_over_match_ids)
    ].sort_values(
        ["match_id", "innings", "overid", "over_ball_num"]
    )[
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "striker_id",
            "striker_name",
            "bowler_id",
            "bowler_name",
            "runs",
            "is_wicket"
        ]
    ].to_string(index=False)
)

' match_id  innings  overid  over_ball_num  striker_id         striker_name  bowler_id        bowler_name  runs  is_wicket\n    40725        1       1              1        7341          Jake McGurk       3298       Jofra Archer     0      False\n    40725        1       1              2        7341          Jake McGurk       3298       Jofra Archer     0      False\n    40725        1       1              3        7341          Jake McGurk       3298       Jofra Archer     4      False\n    40725        1       1              4        7341          Jake McGurk       3298       Jofra Archer     4      False\n    40725        1       1              5        7341          Jake McGurk       3298       Jofra Archer     1      False\n    40725        1       1              6       19714        Abishek Porel       3298       Jofra Archer     1      False\n    40725        1       2              1       19714        Abishek Porel       6806   Tushar Deshpande     4      False\n    40725      

In [11]:
# CHECK ACTUAL DELIVERY COLUMN NAMES

print(ipl_deliveries.columns.tolist())

['match_id', 'innings', 'overid', 'over_ball_num', 'innings_ball_num', 'batting_team_id', 'bowling_team_id', 'striker_id', 'striker_name', 'non_striker_id', 'non_striker_name', 'bowler_id', 'bowler_name', 'batting_type', 'bowling_type', 'batting_order', 'runs', 'is_four', 'is_six', 'is_wicket', 'dismissal_type_id', 'dismissal_type', 'dismissed_player_id', 'extras_type', 'extras1_runs', 'extras_2_runs', 'pace_or_spin', 'bowl_speed', 'stroke', 'delivery_style', 'over_or_around', 'on_or_off', 'delivery_ist_time']


In [12]:
# SEPARATE REGULAR INNINGS FROM SUPER OVERS

ipl_regular_deliveries = ipl_deliveries[
    ipl_deliveries["innings"].isin([1, 2])
].copy()

ipl_super_over_deliveries = ipl_deliveries[
    ipl_deliveries["innings"].isin([3, 4])
].copy()

print("REGULAR IPL DELIVERIES:", len(ipl_regular_deliveries))
print("SUPER OVER DELIVERIES:", len(ipl_super_over_deliveries))

print("\nREGULAR INNINGS DISTRIBUTION:")
display(
    ipl_regular_deliveries.groupby("innings")
    .agg(
        deliveries=("match_id", "size"),
        matches=("match_id", "nunique")
    )
)

REGULAR IPL DELIVERIES: 87673
SUPER OVER DELIVERIES: 14

REGULAR INNINGS DISTRIBUTION:


,deliveries,matches
innings,,
1,45588,367
2,42085,363


In [13]:
# VALIDATE RUNS AND EXTRAS

print("RUNS COLUMN:")
display(ipl_regular_deliveries["runs"].value_counts().sort_index())

print("\nEXTRAS TYPES:")
display(
    ipl_regular_deliveries["extras_type"]
    .value_counts(dropna=False)
)

print("\nEXTRAS COLUMN SUMMARY:")
display(
    ipl_regular_deliveries[
        ["runs", "extras1_runs", "extras_2_runs"]
    ].describe()
)

RUNS COLUMN:


runs
0    33345
1    31941
2     5044
3      193
4    10963
5       13
6     6174
Name: count, dtype: int64


EXTRAS TYPES:


extras_type
NaN         82445
wide         3552
legbyes      1109
Overstep      257
byes          188
other         122
Name: count, dtype: int64


EXTRAS COLUMN SUMMARY:


,runs,extras1_runs,extras_2_runs
count,87673.000000,87673.000000,87673.000000
mean,1.409430,0.072850,0.000536
std,1.779709,0.349019,0.042044
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000
75%,2.000000,0.000000,0.000000
max,6.000000,5.000000,4.000000


In [14]:
# RECREATE IPL DELIVERY DATAFRAMES

ipl_deliveries = ball_by_ball[
    ball_by_ball["match_id"].isin(ipl_matches["match_id"])
].copy()

ipl_regular_deliveries = ipl_deliveries[
    ipl_deliveries["innings"].isin([1, 2])
].copy()

ipl_super_over_deliveries = ipl_deliveries[
    ipl_deliveries["innings"].isin([3, 4])
].copy()

print("IPL deliveries:", len(ipl_deliveries))
print("Regular deliveries:", len(ipl_regular_deliveries))
print("Super over deliveries:", len(ipl_super_over_deliveries))

IPL deliveries: 87687
Regular deliveries: 87673
Super over deliveries: 14


In [15]:
# VALIDATE TOTAL RUNS PER DELIVERY

ipl_regular_deliveries["total_runs"] = (
    ipl_regular_deliveries["runs"]
    + ipl_regular_deliveries["extras1_runs"]
    + ipl_regular_deliveries["extras_2_runs"]
)

print("TOTAL RUNS DISTRIBUTION:")
display(
    ipl_regular_deliveries["total_runs"]
    .value_counts()
    .sort_index()
)

print("\nTOTAL RUNS SUMMARY:")
display(ipl_regular_deliveries["total_runs"].describe())

print("\nDELIVERIES WITH EXTRAS:")
display(
    ipl_regular_deliveries[
        ipl_regular_deliveries["extras_type"].notna()
    ][
        ["match_id", "innings", "overid", "over_ball_num",
         "striker_name", "runs", "extras1_runs",
         "extras_2_runs", "extras_type", "total_runs"]
    ].head(10)
)

TOTAL RUNS DISTRIBUTION:


total_runs
0    28328
1    36372
2     5316
3      216
4    11065
5      202
6     6137
7       37
Name: count, dtype: int64


TOTAL RUNS SUMMARY:


count    87673.000000
mean         1.482817
std          1.760232
min          0.000000
25%          0.000000
50%          1.000000
75%          2.000000
max          7.000000
Name: total_runs, dtype: float64


DELIVERIES WITH EXTRAS:


,match_id,innings,overid,over_ball_num,striker_name,runs,extras1_runs,extras_2_runs,extras_type,total_runs
3191,14094,1,1,1,Ruturaj Gaikwad,0,1,0,Overstep,1
3193,14094,1,1,3,Ruturaj Gaikwad,0,1,0,wide,1
3196,14094,1,1,6,Robin Uthappa,0,1,0,wide,1
3241,14094,1,8,6,Robin Uthappa,0,1,0,wide,1
3306,14094,1,19,4,Ravindra Jadeja,0,1,0,wide,1
3309,14094,1,19,7,Mahendra Singh Dhoni,6,1,0,other,7
3363,14094,2,8,5,Nitish Rana,0,4,0,byes,4
3455,14095,1,5,4,Rohit Sharma,0,1,0,wide,1
3532,14095,1,18,2,Ishan Kishan,0,1,0,legbyes,1
3544,14095,1,20,2,Daniel Sams,0,1,0,wide,1


In [16]:
# CALCULATE IPL INNINGS TOTALS

innings_totals = (
    ipl_regular_deliveries
    .groupby(["match_id", "innings"])
    .agg(
        total_runs=("total_runs", "sum"),
        batter_runs=("runs", "sum"),
        extras_runs=("extras1_runs", "sum"),
        delivery_count=("total_runs", "count")
    )
    .reset_index()
)

print("INNINGS TOTALS:")
display(innings_totals.head(10))

print("\nINNINGS SCORE SUMMARY:")
display(innings_totals["total_runs"].describe())

print("\nNUMBER OF INNINGS:")
print(innings_totals["innings"].value_counts().sort_index())

INNINGS TOTALS:


,match_id,innings,total_runs,batter_runs,extras_runs,delivery_count
0,14094,1,131,125,6,126
1,14094,2,133,129,4,111
2,14095,1,177,174,3,122
3,14095,2,179,168,11,117
4,14096,1,205,182,23,127
5,14096,2,208,186,22,126
6,14097,1,158,154,4,124
7,14097,2,161,152,9,121
8,14098,1,210,196,14,130
9,14098,2,149,139,10,125



INNINGS SCORE SUMMARY:


count    730.000000
mean     178.086301
std       36.273832
min        7.000000
25%      156.000000
50%      178.000000
75%      203.000000
max      287.000000
Name: total_runs, dtype: float64


NUMBER OF INNINGS:
innings
1    367
2    363
Name: count, dtype: int64


In [17]:
# CHECK MATCH INNINGS COMPLETENESS

innings_count = (
    innings_totals
    .groupby("match_id")["innings"]
    .nunique()
    .value_counts()
    .sort_index()
)

print("MATCHES BY NUMBER OF REGULAR INNINGS:")
display(innings_count)

matches_with_missing_innings = (
    innings_totals
    .groupby("match_id")
    .filter(lambda x: x["innings"].nunique() < 2)
)

print("\nMATCHES WITH ONLY ONE REGULAR INNINGS:")
display(matches_with_missing_innings)

print(
    "\nNUMBER OF MATCHES WITH ONLY ONE INNINGS:",
    matches_with_missing_innings["match_id"].nunique()
)

MATCHES BY NUMBER OF REGULAR INNINGS:


innings
1      4
2    363
Name: count, dtype: int64


MATCHES WITH ONLY ONE REGULAR INNINGS:


,match_id,innings,total_runs,batter_runs,extras_runs,delivery_count
238,20226,1,125,119,6,118
545,40748,1,133,128,5,123
550,40751,1,122,120,2,63
605,50136,1,25,25,0,22



NUMBER OF MATCHES WITH ONLY ONE INNINGS: 4


In [19]:
print("IPL MATCH COLUMNS:")
print(ipl_matches.columns.tolist())

IPL MATCH COLUMNS:
['match_id', 'comp_id', 'match_number', 'match_date', 'stage', 'venue_id', 'venue_country', 'home_team_id', 'home_team', 'away_team_id', 'away_team', 'home_team_captain_id', 'away_team_captain_id', 'toss_win_team_id', 'toss_opted', 'man_of_the_match', 'first_inn_revised_overs', 'second_inn_revised_overs', 'revised_target', 'knock_out', 'win_team_id', 'result', 'comp_name', 'match_format', 'season']


In [20]:
# COMPARE CALCULATED SCORES WITH MATCH RESULTS

innings_wide = (
    innings_totals
    .pivot(index="match_id", columns="innings", values="total_runs")
    .reset_index()
)

innings_wide = innings_wide.rename(columns={
    1: "calculated_first_innings",
    2: "calculated_second_innings"
})

score_comparison = innings_wide.merge(
    ipl_matches[
        ["match_id", "home_team_id", "away_team_id", "win_team_id", "result"]
    ],
    on="match_id",
    how="left",
    validate="one_to_one"
)

print("SCORE COMPARISON:")
display(score_comparison.head(10))

print("\nMATCHES WITH MISSING CALCULATED INNINGS SCORES:")
display(
    score_comparison[
        score_comparison[
            ["calculated_first_innings", "calculated_second_innings"]
        ].isna().any(axis=1)
    ]
)

print("\nTOTAL MATCHES:", len(score_comparison))

SCORE COMPARISON:


,match_id,calculated_first_innings,calculated_second_innings,home_team_id,away_team_id,win_team_id,result
0,14094,131.0,133.0,1,8,8.0,Kolkata Knight Riders Won by 6 Wkts
1,14095,177.0,179.0,7,2,7.0,Delhi Capitals Won by 4 Wickets
2,14096,205.0,208.0,4,5,4.0,Punjab Kings Won by 5 Wkts
3,14097,158.0,161.0,3508,3509,3508.0,Gujarat Titans Won by 5 Wickets
4,14098,210.0,149.0,242,3,3.0,Rajasthan Royals Won by 61 Runs
5,14099,128.0,132.0,5,8,5.0,Royal Challengers Bengaluru Won 3 Wkts
6,14100,210.0,211.0,3509,1,3509.0,Lucknow Super Giants Won by 6 Wkts
7,14101,137.0,141.0,8,4,8.0,Kolkata Knight Riders Won by 6 wkts
8,14102,193.0,170.0,2,3,3.0,Rajasthan Royals Won by 23 Runs
9,14103,171.0,157.0,3508,7,3508.0,Gujarat Titans won by 14 runs



MATCHES WITH MISSING CALCULATED INNINGS SCORES:


,match_id,calculated_first_innings,calculated_second_innings,home_team_id,away_team_id,win_team_id,result
119,20226,125.0,NaN,3509,1,NaN,Match Abandoned
273,40748,133.0,NaN,242,7,NaN,No Result
276,40751,122.0,NaN,4,7,NaN,No Result
304,50136,25.0,NaN,8,4,NaN,No Result



TOTAL MATCHES: 367


In [21]:
# IDENTIFY BATTING TEAMS IN EACH INNINGS

batting_teams = (
    ipl_regular_deliveries
    .groupby(["match_id", "innings"])["batting_team_id"]
    .nunique()
    .reset_index(name="batting_team_count")
)

print("BATTING TEAM COUNT PER INNINGS:")
display(batting_teams["batting_team_count"].value_counts())

print("\nINNINGS WITH MORE THAN ONE BATTING TEAM:")
display(
    batting_teams[batting_teams["batting_team_count"] != 1]
)

BATTING TEAM COUNT PER INNINGS:


batting_team_count
1    730
Name: count, dtype: int64


INNINGS WITH MORE THAN ONE BATTING TEAM:


,match_id,innings,batting_team_count


In [22]:
# BUILD INNINGS SCORECARD

innings_scorecard = (
    ipl_regular_deliveries
    .groupby(["match_id", "innings"])
    .agg(
        batting_team_id=("batting_team_id", "first"),
        total_runs=("total_runs", "sum"),
        batter_runs=("runs", "sum"),
        extras_runs=("extras1_runs", "sum"),
        delivery_count=("total_runs", "count")
    )
    .reset_index()
)

print("INNINGS SCORECARD:")
display(innings_scorecard.head(10))

print("\nSCORECARD SHAPE:", innings_scorecard.shape)

print("\nUNIQUE BATTING TEAMS PER INNINGS:")
print(
    innings_scorecard
    .groupby(["match_id", "innings"])["batting_team_id"]
    .nunique()
    .value_counts()
)

INNINGS SCORECARD:


,match_id,innings,batting_team_id,total_runs,batter_runs,extras_runs,delivery_count
0,14094,1,1,131,125,6,126
1,14094,2,8,133,129,4,111
2,14095,1,2,177,174,3,122
3,14095,2,7,179,168,11,117
4,14096,1,5,205,182,23,127
5,14096,2,4,208,186,22,126
6,14097,1,3509,158,154,4,124
7,14097,2,3508,161,152,9,121
8,14098,1,3,210,196,14,130
9,14098,2,242,149,139,10,125



SCORECARD SHAPE: (730, 7)

UNIQUE BATTING TEAMS PER INNINGS:
batting_team_id
1    730
Name: count, dtype: int64


In [23]:
# CREATE MATCH-LEVEL CHASE INFORMATION

first_innings = innings_scorecard[
    innings_scorecard["innings"] == 1
][["match_id", "batting_team_id", "total_runs"]].rename(
    columns={
        "batting_team_id": "first_innings_team_id",
        "total_runs": "first_innings_score"
    }
)

second_innings = innings_scorecard[
    innings_scorecard["innings"] == 2
][["match_id", "batting_team_id", "total_runs"]].rename(
    columns={
        "batting_team_id": "second_innings_team_id",
        "total_runs": "second_innings_score"
    }
)

match_chase = first_innings.merge(
    second_innings,
    on="match_id",
    how="outer",
    validate="one_to_one"
)

match_chase["target"] = match_chase["first_innings_score"] + 1

print("MATCH CHASE INFORMATION:")
display(match_chase.head(10))

print("\nMATCH CHASE DATA SHAPE:", match_chase.shape)

print("\nMATCHES WITHOUT A SECOND INNINGS:")
display(match_chase[match_chase["second_innings_score"].isna()])

MATCH CHASE INFORMATION:


,match_id,first_innings_team_id,first_innings_score,second_innings_team_id,second_innings_score,target
0,14094,1,131,8.0,133.0,132
1,14095,2,177,7.0,179.0,178
2,14096,5,205,4.0,208.0,206
3,14097,3509,158,3508.0,161.0,159
4,14098,3,210,242.0,149.0,211
5,14099,8,128,5.0,132.0,129
6,14100,1,210,3509.0,211.0,211
7,14101,4,137,8.0,141.0,138
8,14102,3,193,2.0,170.0,194
9,14103,3508,171,7.0,157.0,172



MATCH CHASE DATA SHAPE: (367, 6)

MATCHES WITHOUT A SECOND INNINGS:


,match_id,first_innings_team_id,first_innings_score,second_innings_team_id,second_innings_score,target
119,20226,3509,125,NaN,NaN,126
273,40748,7,133,NaN,NaN,134
276,40751,4,122,NaN,NaN,123
304,50136,8,25,NaN,NaN,26


In [24]:
# CLASSIFY CHASE OUTCOMES

match_chase["chase_status"] = np.select(
    [
        match_chase["second_innings_score"].isna(),
        match_chase["second_innings_score"] >= match_chase["target"]
    ],
    [
        "Incomplete",
        "Successful Chase"
    ],
    default="Unsuccessful Chase"
)

print("CHASE OUTCOME DISTRIBUTION:")
display(match_chase["chase_status"].value_counts())

print("\nCHASE OUTCOME BY STATUS:")
display(
    match_chase[
        ["match_id", "target", "second_innings_score", "chase_status"]
    ].head(15)
)

CHASE OUTCOME DISTRIBUTION:


chase_status
Successful Chase      186
Unsuccessful Chase    177
Incomplete              4
Name: count, dtype: int64


CHASE OUTCOME BY STATUS:


,match_id,target,second_innings_score,chase_status
0,14094,132,133.0,Successful Chase
1,14095,178,179.0,Successful Chase
2,14096,206,208.0,Successful Chase
3,14097,159,161.0,Successful Chase
4,14098,211,149.0,Unsuccessful Chase
5,14099,129,132.0,Successful Chase
6,14100,211,211.0,Successful Chase
7,14101,138,141.0,Successful Chase
8,14102,194,170.0,Unsuccessful Chase
9,14103,172,157.0,Unsuccessful Chase


In [25]:
# VALIDATE CHASE OUTCOMES AGAINST OFFICIAL WINNERS

match_chase_validation = match_chase.merge(
    ipl_matches[["match_id", "win_team_id", "result"]],
    on="match_id",
    how="left",
    validate="one_to_one"
)

match_chase_validation["calculated_chase_winner"] = np.where(
    match_chase_validation["chase_status"] == "Successful Chase",
    match_chase_validation["second_innings_team_id"],
    np.where(
        match_chase_validation["chase_status"] == "Unsuccessful Chase",
        match_chase_validation["first_innings_team_id"],
        np.nan
    )
)

match_chase_validation["winner_matches"] = (
    match_chase_validation["calculated_chase_winner"]
    == match_chase_validation["win_team_id"]
)

print("WINNER VALIDATION:")
display(
    match_chase_validation["winner_matches"]
    .value_counts(dropna=False)
)

print("\nMISMATCHED MATCHES:")
display(
    match_chase_validation[
        match_chase_validation["winner_matches"] == False
    ][
        ["match_id", "first_innings_team_id",
         "second_innings_team_id", "first_innings_score",
         "second_innings_score", "win_team_id", "result"]
    ]
)

WINNER VALIDATION:


winner_matches
True     360
False      7
Name: count, dtype: int64


MISMATCHED MATCHES:


,match_id,first_innings_team_id,second_innings_team_id,first_innings_score,second_innings_score,win_team_id,result
119,20226,3509,NaN,125,NaN,NaN,Match Abandoned
147,21586,3508,1.0,214,171.0,1.0,Chennai Super Kings Won by 5 wickets (DLS met...
262,40737,4,8.0,201,7.0,NaN,No Result
273,40748,7,NaN,133,NaN,NaN,No Result
274,40749,2,3508.0,155,147.0,3508.0,Gujarat Titans Won by 3 wickets (DLS Method)
276,40751,4,NaN,122,NaN,NaN,No Result
304,50136,8,NaN,25,NaN,NaN,No Result


In [26]:
# CORRECT WINNER VALIDATION

match_chase_validation["winner_validation"] = np.select(
    [
        match_chase_validation["win_team_id"].isna(),
        match_chase_validation["result"].str.contains(
            "DLS", case=False, na=False
        ),
        match_chase_validation["calculated_chase_winner"]
        == match_chase_validation["win_team_id"]
    ],
    [
        "No Official Winner",
        "DLS Result - Separate Validation Needed",
        "Winner Matches"
    ],
    default="Winner Mismatch"
)

print("CORRECTED VALIDATION SUMMARY:")
display(
    match_chase_validation["winner_validation"]
    .value_counts()
)

print("\nMATCHES REQUIRING INVESTIGATION:")
display(
    match_chase_validation[
        match_chase_validation["winner_validation"] != "Winner Matches"
    ][
        ["match_id", "first_innings_score",
         "second_innings_score", "win_team_id",
         "result", "winner_validation"]
    ]
)

CORRECTED VALIDATION SUMMARY:


winner_validation
Winner Matches                             359
No Official Winner                           5
DLS Result - Separate Validation Needed      3
Name: count, dtype: int64


MATCHES REQUIRING INVESTIGATION:


,match_id,first_innings_score,second_innings_score,win_team_id,result,winner_validation
119,20226,125,NaN,NaN,Match Abandoned,No Official Winner
147,21586,214,171.0,1.0,Chennai Super Kings Won by 5 wickets (DLS met...,DLS Result - Separate Validation Needed
262,40737,201,7.0,NaN,No Result,No Official Winner
273,40748,133,NaN,NaN,No Result,No Official Winner
274,40749,155,147.0,3508.0,Gujarat Titans Won by 3 wickets (DLS Method),DLS Result - Separate Validation Needed
276,40751,122,NaN,NaN,No Result,No Official Winner
304,50136,25,NaN,NaN,No Result,No Official Winner
342,50371,209,203.0,3509.0,Lucknow Super Giants Won by 9 runs (DLS method),DLS Result - Separate Validation Needed


In [27]:
# INSPECT DLS MATCHES

dls_matches = match_chase_validation[
    match_chase_validation["winner_validation"]
    == "DLS Result - Separate Validation Needed"
].copy()

dls_details = dls_matches.merge(
    ipl_matches[
        [
            "match_id",
            "match_date",
            "comp_name",
            "first_inn_revised_overs",
            "second_inn_revised_overs",
            "revised_target",
            "win_team_id",
            "result"
        ]
    ],
    on="match_id",
    how="left",
    suffixes=("", "_official"),
    validate="one_to_one"
)

display(
    dls_details[
        [
            "match_id",
            "match_date",
            "first_innings_score",
            "second_innings_score",
            "target",
            "first_inn_revised_overs",
            "second_inn_revised_overs",
            "revised_target",
            "win_team_id",
            "result"
        ]
    ]
)

,match_id,match_date,first_innings_score,second_innings_score,target,first_inn_revised_overs,second_inn_revised_overs,revised_target,win_team_id,result
0,21586,2023-05-29,214,171.0,215,NaN,90.0,171.0,1.0,Chennai Super Kings Won by 5 wickets (DLS met...
1,40749,2025-05-06,155,147.0,156,NaN,114.0,147.0,3508.0,Gujarat Titans Won by 3 wickets (DLS Method)
2,50371,2026-05-07,209,203.0,210,114.0,114.0,213.0,3509.0,Lucknow Super Giants Won by 9 runs (DLS method)


In [28]:
# Check how revised overs are represented in the dataset

dls_matches = ipl_matches[
    ipl_matches["match_id"].isin([21586, 40749, 50371])
].copy()

display(
    dls_matches[
        [
            "match_id",
            "first_inn_revised_overs",
            "second_inn_revised_overs",
            "revised_target"
        ]
    ]
)

print("\nUnique revised overs values:")
print("First innings:", dls_matches["first_inn_revised_overs"].dropna().unique())
print("Second innings:", dls_matches["second_inn_revised_overs"].dropna().unique())

,match_id,first_inn_revised_overs,second_inn_revised_overs,revised_target
262,21586,NaN,90.0,171.0
581,40749,NaN,114.0,147.0
797,50371,114.0,114.0,213.0



Unique revised overs values:
First innings: [114.]
Second innings: [ 90. 114.]


In [29]:
# Check actual deliveries in the DLS matches

dls_delivery_counts = (
    ipl_regular_deliveries[
        ipl_regular_deliveries["match_id"].isin([21586, 40749, 50371])
    ]
    .groupby(["match_id", "innings"])
    .agg(
        delivery_records=("match_id", "size"),
        legal_balls=("extras_type", lambda x: (~x.eq("wide") & ~x.eq("Overstep")).sum())
    )
    .reset_index()
)

display(dls_delivery_counts)

,match_id,innings,delivery_records,legal_balls
0,21586,1,122,120
1,21586,2,94,90
2,40749,1,121,120
3,40749,2,123,114
4,50371,1,119,114
5,50371,2,122,114


In [30]:
# Create a match-level table with DLS information

dls_match_ids = [21586, 40749, 50371]

ipl_match_summary = ipl_matches.copy()

ipl_match_summary["is_dls"] = (
    ipl_match_summary["match_id"].isin(dls_match_ids)
)

ipl_match_summary["is_no_result"] = (
    ipl_match_summary["win_team_id"].isna()
)

print("Total IPL matches:", len(ipl_match_summary))
print("\nDLS matches:", ipl_match_summary["is_dls"].sum())
print("Matches without official winner:", ipl_match_summary["is_no_result"].sum())

display(
    ipl_match_summary[
        ["match_id", "comp_name", "match_date",
         "is_dls", "is_no_result", "result"]
    ].head(10)
)

Total IPL matches: 371

DLS matches: 3
Matches without official winner: 9


,match_id,comp_name,match_date,is_dls,is_no_result,result
13,14094,IPL 2022,2022-03-26,False,False,Kolkata Knight Riders Won by 6 Wkts
14,14095,IPL 2022,2022-03-27,False,False,Delhi Capitals Won by 4 Wickets
15,14096,IPL 2022,2022-03-27,False,False,Punjab Kings Won by 5 Wkts
16,14097,IPL 2022,2022-03-28,False,False,Gujarat Titans Won by 5 Wickets
17,14098,IPL 2022,2022-03-29,False,False,Rajasthan Royals Won by 61 Runs
18,14099,IPL 2022,2022-03-30,False,False,Royal Challengers Bengaluru Won 3 Wkts
19,14100,IPL 2022,2022-03-31,False,False,Lucknow Super Giants Won by 6 Wkts
20,14101,IPL 2022,2022-04-01,False,False,Kolkata Knight Riders Won by 6 wkts
21,14102,IPL 2022,2022-04-02,False,False,Rajasthan Royals Won by 23 Runs
22,14103,IPL 2022,2022-04-02,False,False,Gujarat Titans won by 14 runs


In [31]:
# IPL match outcomes by season

ipl_match_summary["season"] = (
    ipl_match_summary["comp_name"].str.extract(r"IPL (\d{4})")
)

season_summary = (
    ipl_match_summary
    .groupby("season")
    .agg(
        total_matches=("match_id", "count"),
        matches_with_winner=("win_team_id", "count"),
        no_result_matches=("is_no_result", "sum"),
        dls_matches=("is_dls", "sum")
    )
    .reset_index()
)

display(season_summary)

,season,total_matches,matches_with_winner,no_result_matches,dls_matches
0,2022,74,74,0,0
1,2023,74,73,1,1
2,2024,74,71,3,0
3,2025,75,71,4,1
4,2026,74,73,1,1


In [35]:
# Count IPL matches played by each team

team_participation = pd.concat([
    ipl_match_summary[["home_team_id"]].rename(
        columns={"home_team_id": "team_id"}
    ),
    ipl_match_summary[["away_team_id"]].rename(
        columns={"away_team_id": "team_id"}
    )
])

team_match_counts = (
    team_participation
    .groupby("team_id")
    .size()
    .reset_index(name="matches_played")
    .merge(
        team_lookup,
        on="team_id",
        how="left",
        validate="many_to_one"
    )
    .sort_values("matches_played", ascending=False)
)

display(team_match_counts)

print("Number of distinct teams:", team_match_counts["team_id"].nunique())
print("Missing team names:", team_match_counts["team_name"].isna().sum())

,team_id,matches_played,team_name
8,3508,79,Gujarat Titans
2,3,77,Rajasthan Royals
4,5,77,Royal Challengers Bengaluru
1,2,74,Mumbai Indians
7,242,74,Sunrisers Hyderabad
3,4,74,Punjab Kings
0,1,72,Chennai Super Kings
6,8,72,Kolkata Knight Riders
9,3509,72,Lucknow Super Giants
5,7,71,Delhi Capitals


Number of distinct teams: 10
Missing team names: 0


In [33]:
# Check duplicate team IDs in the teams dataset

print("Total team records:", len(teams))
print("Unique team IDs:", teams["team_id"].nunique())
print("Duplicate team ID records:", teams["team_id"].duplicated().sum())

duplicate_teams = teams[
    teams["team_id"].duplicated(keep=False)
].sort_values("team_id")

display(duplicate_teams)

Total team records: 353
Unique team IDs: 38
Duplicate team ID records: 315


,comp_id,team_id,team_name
79,1715,1,Chennai Super Kings
301,3298,1,Chennai Super Kings
131,2193,1,Chennai Super Kings
209,2725,1,Chennai Super Kings
6,1358,1,Chennai Super Kings
...,...,...,...
88,1715,3509,Lucknow Super Giants
199,2670,3674,Sri Lanka A
299,3289,3674,Sri Lanka A
300,3289,4444,Afghanistan A


In [34]:
# Create a unique team lookup for IPL analysis

team_lookup = (
    teams[["team_id", "team_name"]]
    .drop_duplicates()
    .sort_values("team_id")
)

# Verify that each team ID has only one team name
team_name_counts = (
    team_lookup.groupby("team_id")["team_name"].nunique()
)

print("Unique team records:", len(team_lookup))
print("Unique team IDs:", team_lookup["team_id"].nunique())
print("Team IDs with multiple names:", (team_name_counts > 1).sum())

display(team_lookup.head(10))

Unique team records: 38
Unique team IDs: 38
Team IDs with multiple names: 0


,team_id,team_name
6,1,Chennai Super Kings
7,2,Mumbai Indians
8,3,Rajasthan Royals
9,4,Punjab Kings
10,5,Royal Challengers Bengaluru
11,7,Delhi Capitals
12,8,Kolkata Knight Riders
30,17,New Zealand
2,18,India
19,19,South Africa


In [36]:
# Validate team participation totals

print("Total IPL matches:", len(ipl_match_summary))
print("Expected team appearances:", len(ipl_match_summary) * 2)
print("Actual team appearances:", team_match_counts["matches_played"].sum())

print(
    "Validation:",
    team_match_counts["matches_played"].sum() == len(ipl_match_summary) * 2
)

# Check missing home and away team IDs
print("\nMissing home team IDs:", ipl_match_summary["home_team_id"].isna().sum())
print("Missing away team IDs:", ipl_match_summary["away_team_id"].isna().sum())

Total IPL matches: 371
Expected team appearances: 742
Actual team appearances: 742
Validation: True

Missing home team IDs: 0
Missing away team IDs: 0


In [37]:
# Count official IPL wins by team

ipl_wins = (
    ipl_match_summary[
        ipl_match_summary["win_team_id"].notna()
    ]
    .groupby("win_team_id")
    .size()
    .reset_index(name="matches_won")
    .rename(columns={"win_team_id": "team_id"})
    .merge(
        team_lookup,
        on="team_id",
        how="left",
        validate="one_to_one"
    )
    .sort_values("matches_won", ascending=False)
)

display(ipl_wins)

print("Total official wins:", ipl_wins["matches_won"].sum())
print("Teams with at least one win:", len(ipl_wins))

,team_id,matches_won,team_name
8,3508.0,47,Gujarat Titans
4,5.0,45,Royal Challengers Bengaluru
2,3.0,39,Rajasthan Royals
3,4.0,35,Punjab Kings
7,242.0,34,Sunrisers Hyderabad
6,8.0,34,Kolkata Knight Riders
9,3509.0,34,Lucknow Super Giants
5,7.0,33,Delhi Capitals
0,1.0,31,Chennai Super Kings
1,2.0,30,Mumbai Indians


Total official wins: 362
Teams with at least one win: 10


In [38]:
print(ipl_wins.dtypes)
print("\nActual DataFrame values:")
print(ipl_wins.to_string(index=False))

team_id        float64
matches_won      int64
team_name          str
dtype: object

Actual DataFrame values:
 team_id  matches_won                   team_name
  3508.0           47              Gujarat Titans
     5.0           45 Royal Challengers Bengaluru
     3.0           39            Rajasthan Royals
     4.0           35                Punjab Kings
   242.0           34         Sunrisers Hyderabad
     8.0           34       Kolkata Knight Riders
  3509.0           34        Lucknow Super Giants
     7.0           33              Delhi Capitals
     1.0           31         Chennai Super Kings
     2.0           30              Mumbai Indians


In [39]:
# Inspect winner IDs and team lookup IDs

print("Team lookup:")
print(team_lookup.sort_values("team_id").to_string(index=False))

print("\nWinner IDs:")
print(
    ipl_match_summary["win_team_id"]
    .value_counts(dropna=False)
    .sort_index()
    .to_string()
)

Team lookup:
 team_id                   team_name
       1         Chennai Super Kings
       2              Mumbai Indians
       3            Rajasthan Royals
       4                Punjab Kings
       5 Royal Challengers Bengaluru
       7              Delhi Capitals
       8       Kolkata Knight Riders
      17                 New Zealand
      18                       India
      19                South Africa
      20                   Australia
      21                  Bangladesh
      22                     England
      23                     Ireland
      24                 Netherlands
      25                    Pakistan
      26                    Scotland
      27                   Sri Lanka
      28                 West Indies
      72                    Zimbabwe
      78                 Afghanistan
      89                      Canada
     222                     India A
     242         Sunrisers Hyderabad
     272        United Arab Emirates
     282                 

In [40]:
# Extract team IDs appearing in IPL matches

ipl_team_ids = pd.unique(
    pd.concat([
        ipl_match_summary["home_team_id"],
        ipl_match_summary["away_team_id"]
    ]).dropna()
)

ipl_team_lookup = (
    team_lookup[
        team_lookup["team_id"].isin(ipl_team_ids)
    ]
    .sort_values("team_id")
)

print("Distinct IPL team IDs:", len(ipl_team_ids))
print("\nIPL team ID mapping:")
print(ipl_team_lookup.to_string(index=False))

print("\nWinner IDs missing from IPL team mapping:")

winner_ids = set(
    ipl_match_summary["win_team_id"].dropna().unique()
)

mapped_ids = set(ipl_team_lookup["team_id"].unique())

print(sorted(winner_ids - mapped_ids))

Distinct IPL team IDs: 10

IPL team ID mapping:
 team_id                   team_name
       1         Chennai Super Kings
       2              Mumbai Indians
       3            Rajasthan Royals
       4                Punjab Kings
       5 Royal Challengers Bengaluru
       7              Delhi Capitals
       8       Kolkata Knight Riders
     242         Sunrisers Hyderabad
    3508              Gujarat Titans
    3509        Lucknow Super Giants

Winner IDs missing from IPL team mapping:
[]


In [41]:
# Calculate official IPL wins by team

ipl_wins = (
    ipl_match_summary[
        ipl_match_summary["win_team_id"].notna()
    ]
    .groupby("win_team_id")
    .size()
    .reset_index(name="matches_won")
    .rename(columns={"win_team_id": "team_id"})
    .merge(
        ipl_team_lookup,
        on="team_id",
        how="left",
        validate="one_to_one"
    )
    .sort_values("matches_won", ascending=False)
    .reset_index(drop=True)
)

print(ipl_wins.to_string(index=False))
print("\nTotal official wins:", ipl_wins["matches_won"].sum())

 team_id  matches_won                   team_name
  3508.0           47              Gujarat Titans
     5.0           45 Royal Challengers Bengaluru
     3.0           39            Rajasthan Royals
     4.0           35                Punjab Kings
   242.0           34         Sunrisers Hyderabad
     8.0           34       Kolkata Knight Riders
  3509.0           34        Lucknow Super Giants
     7.0           33              Delhi Capitals
     1.0           31         Chennai Super Kings
     2.0           30              Mumbai Indians

Total official wins: 362


In [42]:
# Calculate team win percentages

team_performance = team_match_counts.merge(
    ipl_wins[["team_id", "matches_won"]],
    on="team_id",
    how="left",
    validate="one_to_one"
)

team_performance["matches_won"] = (
    team_performance["matches_won"].fillna(0).astype(int)
)

team_performance["win_percentage"] = (
    team_performance["matches_won"]
    / team_performance["matches_played"]
    * 100
).round(2)

team_performance = team_performance.sort_values(
    "win_percentage", ascending=False
).reset_index(drop=True)

display(team_performance)

,team_id,matches_played,team_name,matches_won,win_percentage
0,3508,79,Gujarat Titans,47,59.49
1,5,77,Royal Challengers Bengaluru,45,58.44
2,3,77,Rajasthan Royals,39,50.65
3,4,74,Punjab Kings,35,47.30
4,3509,72,Lucknow Super Giants,34,47.22
5,8,72,Kolkata Knight Riders,34,47.22
6,7,71,Delhi Capitals,33,46.48
7,242,74,Sunrisers Hyderabad,34,45.95
8,1,72,Chennai Super Kings,31,43.06
9,2,74,Mumbai Indians,30,40.54


In [43]:
# Calculate win percentage among matches with an official winner

team_performance["decided_matches"] = (
    team_performance["matches_played"]
    - ipl_match_summary["is_no_result"].sum() / 10
)

# Calculate no-result matches separately for each team
no_result_participation = pd.concat([
    ipl_match_summary.loc[
        ipl_match_summary["is_no_result"], "home_team_id"
    ],
    ipl_match_summary.loc[
        ipl_match_summary["is_no_result"], "away_team_id"
    ]
]).value_counts()

team_performance["no_result_matches"] = (
    team_performance["team_id"]
    .map(no_result_participation)
    .fillna(0)
    .astype(int)
)

team_performance["decided_matches"] = (
    team_performance["matches_played"]
    - team_performance["no_result_matches"]
)

team_performance["decided_match_win_percentage"] = (
    team_performance["matches_won"]
    / team_performance["decided_matches"]
    * 100
).round(2)

display(
    team_performance[
        [
            "team_name",
            "matches_played",
            "matches_won",
            "no_result_matches",
            "decided_matches",
            "decided_match_win_percentage"
        ]
    ].sort_values(
        "decided_match_win_percentage",
        ascending=False
    )
)

,team_name,matches_played,matches_won,no_result_matches,decided_matches,decided_match_win_percentage
0,Gujarat Titans,79,47,2,77,61.04
1,Royal Challengers Bengaluru,77,45,1,76,59.21
2,Rajasthan Royals,77,39,1,76,51.32
5,Kolkata Knight Riders,72,34,5,67,50.75
3,Punjab Kings,74,35,3,71,49.30
4,Lucknow Super Giants,72,34,1,71,47.89
6,Delhi Capitals,71,33,2,69,47.83
7,Sunrisers Hyderabad,74,34,2,72,47.22
8,Chennai Super Kings,72,31,1,71,43.66
9,Mumbai Indians,74,30,0,74,40.54


In [45]:
# Analyse wins by batting first or chasing

match_chase_results = match_chase_validation.copy()

# Add DLS and no-result flags without duplicating winner columns
match_chase_results = match_chase_results.merge(
    ipl_match_summary[
        ["match_id", "is_dls", "is_no_result"]
    ],
    on="match_id",
    how="left",
    validate="one_to_one"
)

match_chase_results["win_type"] = np.select(
    [
        match_chase_results["is_no_result"],
        match_chase_results["win_team_id"]
        == match_chase_results["first_innings_team_id"],
        match_chase_results["win_team_id"]
        == match_chase_results["second_innings_team_id"]
    ],
    [
        "No Result",
        "Batting First",
        "Chasing"
    ],
    default="Other / Requires Review"
)

display(match_chase_results["win_type"].value_counts())

win_type
Chasing          188
Batting First    174
No Result          5
Name: count, dtype: int64

In [46]:
# Inspect super-over match outcomes

super_over_match_ids = [40725, 50359]

super_over_results = match_chase_results[
    match_chase_results["match_id"].isin(super_over_match_ids)
][
    [
        "match_id",
        "first_innings_team_id",
        "second_innings_team_id",
        "first_innings_score",
        "second_innings_score",
        "win_team_id",
        "result",
        "win_type"
    ]
]

display(super_over_results)

,match_id,first_innings_team_id,second_innings_team_id,first_innings_score,second_innings_score,win_team_id,result,win_type
250,40725,7,3.0,188,188.0,7.0,Delhi Capitals Tied with Rajasthan Royals (DC ...,Batting First
330,50359,8,3509.0,155,155.0,8.0,Kolkata Knight Riders Won by Super over (Match...,Batting First


In [47]:
# Inspect super-over results without table formatting

print(
    super_over_results.to_string(index=False)
)

print("\nColumn data types:")
print(super_over_results.dtypes)

 match_id  first_innings_team_id  second_innings_team_id  first_innings_score  second_innings_score  win_team_id                                                            result      win_type
    40725                      7                     3.0                  188                 188.0          7.0 Delhi Capitals Tied with Rajasthan Royals (DC won the Super Over) Batting First
    50359                      8                  3509.0                  155                 155.0          8.0              Kolkata Knight Riders Won by Super over (Match Tied) Batting First

Column data types:
match_id                    int64
first_innings_team_id       int64
second_innings_team_id    float64
first_innings_score         int64
second_innings_score      float64
win_team_id               float64
result                        str
win_type                      str
dtype: object


In [48]:
# Classify match outcomes, including Super Over matches

super_over_match_ids = [40725, 50359]

match_chase_results["win_type"] = np.select(
    [
        match_chase_results["is_no_result"],
        match_chase_results["match_id"].isin(super_over_match_ids),
        match_chase_results["win_team_id"]
        == match_chase_results["first_innings_team_id"],
        match_chase_results["win_team_id"]
        == match_chase_results["second_innings_team_id"]
    ],
    [
        "No Result",
        "Super Over Win",
        "Batting First",
        "Chasing"
    ],
    default="Other / Requires Review"
)

display(match_chase_results["win_type"].value_counts())

win_type
Chasing           188
Batting First     172
No Result           5
Super Over Win      2
Name: count, dtype: int64

In [49]:
# Inspect DLS match classifications

dls_results = match_chase_results[
    match_chase_results["is_dls"] == True
][
    [
        "match_id",
        "first_innings_team_id",
        "second_innings_team_id",
        "first_innings_score",
        "second_innings_score",
        "win_team_id",
        "result",
        "win_type"
    ]
]

print(dls_results.to_string(index=False))

 match_id  first_innings_team_id  second_innings_team_id  first_innings_score  second_innings_score  win_team_id                                             result      win_type
    21586                   3508                     1.0                  214                 171.0          1.0 Chennai Super Kings Won by 5 wickets  (DLS method)       Chasing
    40749                      2                  3508.0                  155                 147.0       3508.0       Gujarat Titans Won by 3 wickets (DLS Method)       Chasing
    50371                   3509                     5.0                  209                 203.0       3509.0    Lucknow Super Giants Won by 9 runs (DLS method) Batting First


In [50]:
# Final match outcome summary

final_outcome_summary = (
    match_chase_results["win_type"]
    .value_counts()
    .rename_axis("match_outcome")
    .reset_index(name="match_count")
)

final_outcome_summary["percentage"] = (
    final_outcome_summary["match_count"]
    / len(match_chase_results)
    * 100
).round(2)

display(final_outcome_summary)

print("\nTotal classified matches:", final_outcome_summary["match_count"].sum())

,match_outcome,match_count,percentage
0,Chasing,188,51.23
1,Batting First,172,46.87
2,No Result,5,1.36
3,Super Over Win,2,0.54



Total classified matches: 367


In [51]:
# Save match analysis outputs

OUTPUT_PATH = "../outputs/match_analysis/"
import os

os.makedirs(OUTPUT_PATH, exist_ok=True)

ipl_match_summary.to_csv(
    OUTPUT_PATH + "ipl_match_summary.csv",
    index=False
)

innings_scorecard.to_csv(
    OUTPUT_PATH + "ipl_innings_scorecard.csv",
    index=False
)

match_chase_results.to_csv(
    OUTPUT_PATH + "ipl_match_chase_results.csv",
    index=False
)

team_performance.to_csv(
    OUTPUT_PATH + "ipl_team_performance.csv",
    index=False
)

final_outcome_summary.to_csv(
    OUTPUT_PATH + "ipl_match_outcome_summary.csv",
    index=False
)

print("Match analysis outputs saved successfully.")

Match analysis outputs saved successfully.
